# Solar & Wind Atlas catalog explorer (offline)

Inspect the bundled layer catalog without touching the network: which layers
ship, which atlas each comes from, and the transport used to fetch it. This is
the no-network companion to the live wind quickstart.


In [ ]:
import pandas as pd

from earthlens.solar_wind_atlas import Catalog

## The shipped layers

`Catalog()` loads the bundled sharded `catalog/` (`solar.yaml` + `wind.yaml`).
Each row is a frozen pydantic `Layer` carrying its atlas, transport, URL, and units.


In [ ]:
catalog = Catalog()
rows = [
    {
        "id": lid,
        "atlas": catalog.get(lid).atlas,
        "transport": catalog.get(lid).transport,
        "units": catalog.get(lid).units,
        "long_name": catalog.get(lid).long_name,
    }
    for lid in catalog.available()
]
pd.DataFrame(rows)

## Two atlases, two transports

- **Global Wind Atlas** layers (`atlas='gwa'`) use the `vsicurl` transport: a
  windowed read straight from the remote Cloud-Optimized GeoTIFF — only the AOI
  transfers.
- **Global Solar Atlas** layers (`atlas='gsa'`) use `download_zip`: the deflate
  ZIP archive (no random access) is downloaded once into a cache, then cropped
  locally.


In [ ]:
by_transport = {}
for lid in catalog.available():
    by_transport.setdefault(catalog.get(lid).transport, []).append(lid)
by_transport

## Did-you-mean on an unknown id

An unknown layer id raises a `ValueError` with the closest match.


In [ ]:
catalog.get("gho")  # unknown id -> ValueError with a did-you-mean hint